# 1D Heat Equation and Gaussian Scale-Space

The **heat equation** (diffusion equation) in 1D is:
$$
\partial_t u(x, t) = \partial_{xx} u(x, t), \qquad u(x, 0) = f(x).
$$
Its solution is given by **convolution with a Gaussian kernel**:
$$
u(x, t) = (G_t * f)(x) = \frac{1}{\sqrt{4\pi t}} \int_{-\infty}^\infty f(y) e^{-(x-y)^2/(4t)} \, dy.
$$

## Scale-space

The family $\{u(\cdot, t)\}_{t \ge 0}$ forms a **scale-space** representation of $f$:
- At $t=0$: the original signal.
- As $t$ increases: progressively smoother (coarser) versions.
- The Gaussian $G_t$ is the **only** smoothing kernel that satisfies certain natural axioms (non-creation of new extrema, causality, isotropy).

## Semi-group property

Gaussian convolution satisfies $G_s * G_t = G_{s+t}$, so the heat equation semigroup acts on $f$ as a one-parameter group:
$$
u(\cdot, s+t) = G_s * u(\cdot, t).
$$

## Fourier analysis

In Fourier space, the heat equation becomes $\partial_t \hat{u}(\omega, t) = -\omega^2 \hat{u}$, so:
$$
\hat{u}(\omega, t) = e^{-\omega^2 t} \hat{f}(\omega).
$$
High frequencies decay exponentially fast; low frequencies survive.

## Environment

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatLogSlider, IntSlider

plt.rcParams['figure.dpi'] = 120

## Gaussian convolution implementation

We implement the heat equation solution via FFT-based convolution.

In [ ]:
N = 512
x = np.linspace(-5, 5, N)
dx = x[1] - x[0]

def heat_1d(f, t):
    """Convolve f with G_t = exp(-x^2/(4t)) / sqrt(4*pi*t) via FFT."""
    if t <= 0: return f.copy()
    sigma = np.sqrt(2 * t)  # Gaussian std
    freqs = np.fft.rfftfreq(N, d=dx)
    F_hat = np.fft.rfft(f)
    filt = np.exp(-2 * np.pi**2 * freqs**2 * sigma**2)
    return np.fft.irfft(F_hat * filt, n=N)

# Signal: sum of Gaussians with different widths
f0 = (np.exp(-5*(x-0.5)**2) + 0.6*np.exp(-20*(x+1.5)**2) +
      0.3*np.sin(8*x) * np.exp(-x**2))

t_values = [0, 0.02, 0.1, 0.5, 2.0]
cols = plt.cm.plasma(np.linspace(0.1, 0.9, len(t_values)))

fig, ax = plt.subplots(figsize=(10, 4.5))
for t, col in zip(t_values, cols):
    u = heat_1d(f0, t)
    ax.plot(x, u, color=col, lw=2, label=f'$t={t}$')
ax.set_xlabel('$x$'); ax.set_ylabel('$u(x,t)$')
ax.set_title('Heat equation solution $u(x,t) = G_t * f$')
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Scale-space representation

We display the evolution as a 2D image: time on the vertical axis, position on the horizontal axis. This is the **scale-space stack**.

In [ ]:
t_log = np.logspace(-3, 1, 100)
u_stack = np.array([heat_1d(f0, t) for t in t_log])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
im = axes[0].imshow(u_stack, aspect='auto', cmap='RdBu_r',
                    extent=[x[0], x[-1], np.log10(t_log[-1]), np.log10(t_log[0])],
                    origin='upper')
plt.colorbar(im, ax=axes[0], fraction=0.046)
axes[0].set_xlabel('$x$'); axes[0].set_ylabel('$\\log_{10}(t)$')
axes[0].set_title('Scale-space $u(x,t)$')

# Fourier magnitude at different times
for t, col in zip(t_values, cols):
    u = heat_1d(f0, t)
    F = np.abs(np.fft.rfft(u))
    freqs = np.fft.rfftfreq(N, d=dx)
    axes[1].semilogy(freqs[:N//4], F[:N//4] + 1e-10, color=col, lw=2, label=f'$t={t}$')
axes[1].set_xlabel('frequency $\\omega$'); axes[1].set_ylabel('$|\\hat{u}|$ (log scale)')
axes[1].set_title('Fourier spectrum: high frequencies decay as $e^{-\\omega^2 t}$')
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Semi-group property verification

We verify $G_s * (G_t * f) = G_{s+t} * f$.

In [ ]:
s, t_v = 0.1, 0.3
u_st = heat_1d(heat_1d(f0, s), t_v)
u_spt = heat_1d(f0, s + t_v)
print(f'Max difference |G_s*(G_t*f) - G_{{s+t}}*f|: {np.max(np.abs(u_st - u_spt)):.2e}')

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(x, u_st, 'royalblue', lw=2, label=f'$G_{{{s}}} * (G_{{{t_v}}} * f)$')
ax.plot(x, u_spt, 'tomato', ls='--', lw=2, label=f'$G_{{{s+t_v}}} * f$')
ax.set_xlabel('$x$'); ax.set_title('Semigroup property verification'); ax.legend()
ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

## Interactive: diffusion time

In [ ]:
def show_heat(log_t=-2.0, n_signals=1):
    t_i = 10**log_t
    u_i = heat_1d(f0, t_i)
    # Add a piecewise-constant signal for comparison
    f_pc = np.zeros(N)
    f_pc[100:200] = 1.0; f_pc[300:380] = 0.6
    u_pc = heat_1d(f_pc, t_i)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    axes[0].plot(x, f0, 'k-', lw=1, alpha=0.4, label='original')
    axes[0].plot(x, u_i, 'royalblue', lw=2.5, label=f'$t={t_i:.4f}$')
    axes[0].set_title('Smooth signal'); axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].plot(x, f_pc, 'k-', lw=1, alpha=0.4, label='original')
    axes[1].plot(x, u_pc, 'tomato', lw=2.5, label=f'$t={t_i:.4f}$')
    axes[1].set_title('Piecewise-constant signal'); axes[1].legend(); axes[1].grid(alpha=0.3)
    plt.tight_layout(); plt.show()

interact(show_heat,
         log_t=FloatLogSlider(value=-2.0, min=-3.0, max=1.0, step=0.25, description='$\\log_{10}t$'),
         n_signals=IntSlider(value=1, min=1, max=3, step=1, description='signals'));

## Bibliographical resources

- Lindeberg, T. (1994). *Scale-Space Theory in Computer Vision*. Kluwer.
- Witkin, A. (1983). Scale-space filtering. *Proceedings of the 8th International Joint Conference on Artificial Intelligence*, 1019–1022.
- Evans, L. C. (2010). *Partial Differential Equations* (2nd ed.). American Mathematical Society.
- Weickert, J. (1998). *Anisotropic Diffusion in Image Processing*. Teubner.
- Koenderink, J. J. (1984). The structure of images. *Biological Cybernetics*, 50(5), 363–370.